# 02c · I device
*Blocco 2 · IoT Hub con ADR e DPS. Passo di creazione.*

I **sei device**: i quattro della flotta, più `device-da-aggiornare` e `device-da-revocare`, uno per ogni demo che li cambia o li distrugge. Per ognuno la chiave e la richiesta di certificato (CSR) nascono sul device, poi la registrazione a DPS e il **certificato operativo emesso da ADR**. Poi le proprietà (produttore, modello, software revision) e la verifica.

**Prima:** i notebook 01, 02a e 02b.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Fabric: valori dalla Eventstream creata dal 17a (417bd75b-4747-4eff-9d6e-d00ca5964001)

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## 5 · I device
Per ogni device, quattro passaggi: **chiave e CSR nascono sul device** (la chiave non esce); un **certificato di bootstrap**, firmato dalla
CA della flotta, serve **solo a presentarsi a DPS**; DPS inoltra la CSR ad ADR; la **Issuing CA** firma e DPS restituisce la catena.
Il certificato operativo è quello con cui il device parlerà con l'Hub.

La funzione sotto contiene i quattro passaggi; poi la si chiama una volta per device.

## Cosa va dove, in produzione
Nella demo tutto sta sullo stesso disco, e la stessa macchina fa da fabbrica, da device e da operatore. In produzione i ruoli sono separati, e i segreti sono tre:

| File | Dove va custodito | Segreto? |
|---|---|---|
| `ca-key.pem` (CA di bootstrap) | In un vault o HSM di chi produce i device. **Mai sul device** | **Sì** |
| `ca.pem` (CA di bootstrap) | Si carica su DPS e si verifica. Il device non ne ha bisogno | No |
| `bootstrap-key.pem` | Sul device, caricata alla produzione, in uno storage protetto | **Sì** |
| `bootstrap-cert.pem`, `bootstrap-chain.pem` | Sul device, accanto alla sua chiave | No |
| `device-key.pem` | Nasce e resta sul device (TPM o secure element), mai esportata | **Sì** |
| `device.csr.pem` | Il device la manda a DPS; non serve conservarla | No |
| `issued-chain.pem` | Il device la riceve da DPS e la tiene con la sua chiave | No |

Le indicazioni sul luogo sono buone pratiche: la demo non le applica. La cartella `artifacts` contiene chiavi private e non si pubblica.

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
# La funzione Register-DemoDevice sta nella libreria (AzureADRIoTHubLib\AdrDps.ps1): get-help Register-DemoDevice -Full
$results = @()

### robot-01

In [ ]:
$results += Register-DemoDevice -Device ($devices | Where-Object Id -eq 'robot-01') -OutDir $out -DpsName $dps.Name -ResourceGroup $rg

[robot-01] 1. chiave ECC P-384 e CSR con CN = registration ID

[robot-01] 2. certificato di bootstrap firmato dalla CA della flotta (leaf + CA nel file di catena)

[robot-01] 3. registrazione a DPS: il certificato di bootstrap autentica, la CSR chiede il certificato operativo

[robot-01] 4. Hub assegnato: iot-iot26-dev-uks-001.azure-devices.net | certificati nella catena emessa: 3

### robot-02

In [ ]:
$results += Register-DemoDevice -Device ($devices | Where-Object Id -eq 'robot-02') -OutDir $out -DpsName $dps.Name -ResourceGroup $rg

### truck-01

In [ ]:
$results += Register-DemoDevice -Device ($devices | Where-Object Id -eq 'truck-01') -OutDir $out -DpsName $dps.Name -ResourceGroup $rg

### truck-02

In [ ]:
$results += Register-DemoDevice -Device ($devices | Where-Object Id -eq 'truck-02') -OutDir $out -DpsName $dps.Name -ResourceGroup $rg

### device-da-aggiornare
Nel gruppo **robots**, quindi sul primo Hub: è il device della demo degli aggiornamenti (notebook 15).

In [ ]:
$results += Register-DemoDevice -Device ($devices | Where-Object Id -eq 'device-da-aggiornare') -OutDir $out -DpsName $dps.Name -ResourceGroup $rg

### device-da-revocare
Nel gruppo **trucks**, quindi sul secondo Hub: è il device della demo della revoca dei certificati (notebook 09). Nessun'altra demo lo usa.

In [ ]:
$results += Register-DemoDevice -Device ($devices | Where-Object Id -eq 'device-da-revocare') -OutDir $out -DpsName $dps.Name -ResourceGroup $rg

In [ ]:
$results | Select-Object Device, Gruppo, Hub, Stato, Catena | Format-Table -AutoSize | Out-String

## 6 · Chi è il device: manufacturer, model, software revision
Nella pagina del device (portale, sezione *General*) sono i tre campi modificabili. Li imposta `az iot adr ns device update`: **proprietà** del device, non attributi. *Manufacturer* e *model* descrivono il device; la *software revision* è la versione installata ed è il valore che un aggiornamento cambia (notebook 14). Le tre proprietà si possono usare nella query di un gruppo; la query vuole `*` oppure una clausola `WHERE` (notebook 15).

In [ ]:
foreach ($d in $devices) {
    az iot adr ns device update --name $d.Id --namespace $adr.Name --resource-group $rg --manufacturer $d.Manufacturer --model $d.Model --software-revision $d.SoftwareRevision -o none --only-show-errors
    Assert-Az "Proprieta' del device $($d.Id)"
}
az iot adr ns device list --namespace $adr.Name --resource-group $rg --query "[].{device:name, manufacturer:properties.manufacturer, model:properties.model, software:properties.softwareRevision}" -o table

Device                Manufacturer    Model    Software

--------------------  --------------  -------  ----------

truck-02              Contoso         TruckV1  1.0.0

truck-01              Contoso         TruckV1  1.0.0

device-da-revocare    Contoso         TruckV1  1.0.0

robot-02              Contoso         RobotV1  1.0.0

robot-01              Contoso         RobotV1  1.0.0

device-da-aggiornare  Contoso         RobotV1  1.0.0

## 7 · Verifica
**Una vista sola**: i device come risorse del namespace ADR, indipendentemente dall'Hub. Poi i device in ogni Hub (serve il ruolo
*IoT Hub Data Contributor* sul singolo Hub) e la **scadenza dei certificati emessi**: la policy decide la durata, il rinnovo lo avvia
il device (notebook 05).

In [ ]:
az iot adr ns device list --namespace $adr.Name --resource-group $rg -o table

In [ ]:
foreach ($h in $hubs) {
    "--- $($h.Name) ($($h.Region))"
    Grant-HubDataAccess -HubName $h.Name -ResourceGroup $rg
    az iot hub device-identity list --hub-name $h.Name --resource-group $rg --auth-type login --query '[].{device:deviceId, stato:status, autenticazione:authentication.type}' -o table --only-show-errors
}

In [ ]:
$rows = foreach ($d in $devices) {
    $latest = Get-ChildItem (Join-Path $out "devices\$($d.Id)") -Directory -ErrorAction SilentlyContinue | Sort-Object Name | Select-Object -Last 1
    $chain = if ($latest) { Join-Path $latest.FullName 'issued-chain.pem' }
    if ($chain -and (Test-Path $chain)) {
        $s = Get-CertSummary -Path $chain
        [pscustomobject]@{ Device = $d.Id; Soggetto = $s.Subject; Emittente = $s.Issuer; Scade = $s.NotAfter.ToString('yyyy-MM-dd'); Giorni = $s.Days }
    }
    else { [pscustomobject]@{ Device = $d.Id; Soggetto = '(nessuna catena emessa)'; Emittente = ''; Scade = ''; Giorni = '' } }
}
$rows | Format-Table -AutoSize | Out-String

Prossimo notebook: **03**, i file del device.